# VL06IF | Monitor de remessas | Validação do recorte SAP no Datalake

**Tabela:** `dev_procurement.corp_curated.vw_ds_log_monitor_remessas`  
**Transação SAP:** `VL06IF`  
**Objeto:** **VIEW**  
**Colunas:** 41  
**Critério do recorte:** lista de remessas extraída do SAP  
**Quantidade esperada:** 5.331 remessas distintas  
**Linguagem:** SQL  

---

## Objetivo

Validar no Datalake o mesmo universo de remessas extraído da transação SAP VL06IF.

O notebook deverá:

1. interpretar a lista de remessas copiada do Excel;
2. validar a quantidade de IDs informados;
3. eliminar cabeçalho, linhas vazias e duplicidades;
4. normalizar zeros à esquerda somente na chave auxiliar;
5. localizar as remessas na view do Datalake;
6. calcular cobertura de registros;
7. listar todas as remessas não encontradas;
8. verificar duplicidades no recorte do Datalake;
9. analisar o preenchimento das 41 colunas;
10. identificar campos 100% nulos;
11. analisar datas, centros, fornecedores, materiais e status;
12. gerar o recorte completo para exportação e comparação externa com o SAP.

---

## Escopo e limitação

Os arquivos SAP completos não estão disponíveis dentro do Databricks.

O notebook recebe somente a coluna `Remessa`, copiada diretamente do Excel.

Por isso, o notebook permite validar:

- cobertura das remessas;
- remessas ausentes no Datalake;
- duplicidades;
- granularidade do recorte;
- preenchimento das colunas;
- domínios e distribuições internas;
- consistência estrutural da view;
- extração das 41 colunas para comparação externa.

O notebook não permite, sozinho, comparar os valores SAP de fornecedor, peso, material, quantidade, datas e status, porque esses valores não foram carregados no Databricks.

---

## Como usar

Execute as células de cima para baixo.

As células criam `TEMP VIEW` e, portanto, dependem da mesma sessão SQL.

Se o SQL Warehouse ou cluster for reiniciado, execute novamente desde a primeira célula SQL.

Na célula de entrada, copie a coluna inteira do Excel entre a primeira e a última aspa simples.

Formato aceito:

## 1. Metadados

Este objeto é uma **view**. `DESCRIBE DETAIL` e `DESCRIBE HISTORY` só funcionam em tabelas
Delta — por isso ficaram comentados, para não interromper o **Run All**.

Se precisar do histórico, rode-os na tabela física de origem.

In [0]:
%python
-- ============================================================ -- SEÇÃO 1 -- Entrada da lista de remessas extraída do SAP -- -- Cole a coluna inteira do Excel entre as aspas simples. -- Não adicione vírgulas, parênteses ou aspas individuais. -- ============================================================ CREATE OR REPLACE TEMP VIEW vl06if_texto_sap AS SELECT 'num_remessa 1012248445 1012248446 1012248447 1012248448 1012248449 1012248450 1012248451 1012248452 1012248453 1012248454 1012248455 COLE AQUI TODAS AS DEMAIS REMESSAS 1012256290 1012256291 1012256292 1012256298 1012256308' AS lista_remessas;

In [0]:
-- 1.1 DETALHE E HISTORICO — nao se aplicam a view.
-- Descomente somente se o objeto for convertido em tabela Delta.
-- DESCRIBE DETAIL dev_procurement.corp_curated.vw_ds_log_monitor_remessas;
-- DESCRIBE HISTORY dev_procurement.corp_curated.vw_ds_log_monitor_remessas LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                        AS linhas,
       COUNT(DISTINCT CONCAT_WS('|', `num_remessa`, `cod_centro`)) AS chaves_completas,
       COUNT(DISTINCT `cod_centro`)                 AS valores_de_cod_centro,
       COUNT(DISTINCT `num_remessa`)                 AS num_remessa_distintos
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas)
que contenha os casos-limite identificados nas demais seções.

In [0]:
-- 3. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional multiplicando linhas.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas),
g AS (
  SELECT 'num_remessa' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_remessa` FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas)
UNION ALL
  SELECT 'num_remessa + num_item_remessa' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_remessa`, `num_item_remessa` FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas)
UNION ALL
  SELECT 'num_remessa + cod_centro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_remessa`, `cod_centro` FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_remessa + num_item_remessa`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_remessa`, `num_item_remessa`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY `num_remessa`, `num_item_remessa`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (
  SELECT `num_remessa`, `num_item_remessa` FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `num_remessa`, `num_item_remessa` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas c JOIN dup ON c.`num_remessa` <=> dup.`num_remessa` AND c.`num_item_remessa` <=> dup.`num_item_remessa`
),
agg AS (
  SELECT `num_remessa`, `num_item_remessa`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `nm_material_cliente`) AS `nm_material_cliente`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_fornecedor`) AS `nm_fornecedor`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `sg_unidade_medida_basica`) AS `sg_unidade_medida_basica`,
         COUNT(DISTINCT `qt_fornecida_fato_umv`) AS `qt_fornecida_fato_umv`,
         COUNT(DISTINCT `num_pedido_compra`) AS `num_pedido_compra`,
         COUNT(DISTINCT `nm_inverted`) AS `nm_inverted`,
         COUNT(DISTINCT `qt_total_volumes_remessa`) AS `qt_total_volumes_remessa`,
         COUNT(DISTINCT `qt_itens_relevantes`) AS `qt_itens_relevantes`,
         COUNT(DISTINCT `cod_local_expedicao_ponto_recebimento`) AS `cod_local_expedicao_ponto_recebimento`,
         COUNT(DISTINCT `num_conhecimento_transporte`) AS `num_conhecimento_transporte`,
         COUNT(DISTINCT `qt_peso_bruto_total`) AS `qt_peso_bruto_total`,
         COUNT(DISTINCT `st_verificacao_credito_global`) AS `st_verificacao_credito_global`,
         COUNT(DISTINCT `ind_correcao_re_avancada`) AS `ind_correcao_re_avancada`,
         COUNT(DISTINCT `qt_contador`) AS `qt_contador`,
         COUNT(DISTINCT `ind_item_devolucao`) AS `ind_item_devolucao`,
         COUNT(DISTINCT `tp_remessa`) AS `tp_remessa`,
         COUNT(DISTINCT `tp_ctg_item_remessa`) AS `tp_ctg_item_remessa`,
         COUNT(DISTINCT `tp_movimento_administracao_estoques`) AS `tp_movimento_administracao_estoques`,
         COUNT(DISTINCT `st_movimento_mercadorias_item`) AS `st_movimento_mercadorias_item`,
         COUNT(DISTINCT `dt_remessa`) AS `dt_remessa`,
         COUNT(DISTINCT `dt_movimentacao_mercadoria_real`) AS `dt_movimentacao_mercadoria_real`,
         COUNT(DISTINCT `dt_planejada_movimento_mercadoria`) AS `dt_planejada_movimento_mercadoria`,
         COUNT(DISTINCT `dt_disponibilizacao_material`) AS `dt_disponibilizacao_material`,
         COUNT(DISTINCT `num_item_pedido_compra`) AS `num_item_pedido_compra`,
         COUNT(DISTINCT `num_referencia`) AS `num_referencia`,
         COUNT(DISTINCT `st_movimento_mercadorias_todos_itens`) AS `st_movimento_mercadorias_todos_itens`,
         COUNT(DISTINCT `cod_zonas_disponibilizacao_sistema_deposito`) AS `cod_zonas_disponibilizacao_sistema_deposito`,
         COUNT(DISTINCT `cod_organizacao_vendas`) AS `cod_organizacao_vendas`,
         COUNT(DISTINCT `cod_canal_distribuicao`) AS `cod_canal_distribuicao`,
         COUNT(DISTINCT `cod_setor_atividade_organizacao`) AS `cod_setor_atividade_organizacao`,
         COUNT(DISTINCT `tp_equipe_vendas`) AS `tp_equipe_vendas`,
         COUNT(DISTINCT `cod_emissor_ordem`) AS `cod_emissor_ordem`,
         COUNT(DISTINCT `dh_atualizacao`) AS `dh_atualizacao`,
         COUNT(DISTINCT `dh_carga`) AS `dh_carga`
  FROM d GROUP BY `num_remessa`, `num_item_remessa`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(39,
    'cod_centro', MAX(`cod_centro`),
    'cod_material', MAX(`cod_material`),
    'nm_material_cliente', MAX(`nm_material_cliente`),
    'dt_criacao', MAX(`dt_criacao`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_fornecedor', MAX(`nm_fornecedor`),
    'cod_deposito', MAX(`cod_deposito`),
    'sg_unidade_medida_basica', MAX(`sg_unidade_medida_basica`),
    'qt_fornecida_fato_umv', MAX(`qt_fornecida_fato_umv`),
    'num_pedido_compra', MAX(`num_pedido_compra`),
    'nm_inverted', MAX(`nm_inverted`),
    'qt_total_volumes_remessa', MAX(`qt_total_volumes_remessa`),
    'qt_itens_relevantes', MAX(`qt_itens_relevantes`),
    'cod_local_expedicao_ponto_recebimento', MAX(`cod_local_expedicao_ponto_recebimento`),
    'num_conhecimento_transporte', MAX(`num_conhecimento_transporte`),
    'qt_peso_bruto_total', MAX(`qt_peso_bruto_total`),
    'st_verificacao_credito_global', MAX(`st_verificacao_credito_global`),
    'ind_correcao_re_avancada', MAX(`ind_correcao_re_avancada`),
    'qt_contador', MAX(`qt_contador`),
    'ind_item_devolucao', MAX(`ind_item_devolucao`),
    'tp_remessa', MAX(`tp_remessa`),
    'tp_ctg_item_remessa', MAX(`tp_ctg_item_remessa`),
    'tp_movimento_administracao_estoques', MAX(`tp_movimento_administracao_estoques`),
    'st_movimento_mercadorias_item', MAX(`st_movimento_mercadorias_item`),
    'dt_remessa', MAX(`dt_remessa`),
    'dt_movimentacao_mercadoria_real', MAX(`dt_movimentacao_mercadoria_real`),
    'dt_planejada_movimento_mercadoria', MAX(`dt_planejada_movimento_mercadoria`),
    'dt_disponibilizacao_material', MAX(`dt_disponibilizacao_material`),
    'num_item_pedido_compra', MAX(`num_item_pedido_compra`),
    'num_referencia', MAX(`num_referencia`),
    'st_movimento_mercadorias_todos_itens', MAX(`st_movimento_mercadorias_todos_itens`),
    'cod_zonas_disponibilizacao_sistema_deposito', MAX(`cod_zonas_disponibilizacao_sistema_deposito`),
    'cod_organizacao_vendas', MAX(`cod_organizacao_vendas`),
    'cod_canal_distribuicao', MAX(`cod_canal_distribuicao`),
    'cod_setor_atividade_organizacao', MAX(`cod_setor_atividade_organizacao`),
    'tp_equipe_vendas', MAX(`tp_equipe_vendas`),
    'cod_emissor_ordem', MAX(`cod_emissor_ordem`),
    'dh_atualizacao', MAX(`dh_atualizacao`),
    'dh_carga', MAX(`dh_carga`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Em validação anterior, esta análise revelou 7 colunas 100% nulas no Datalake —
uma delas preenchida em 97,9% dos registros do SAP. Este erro **não aparece** em amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, a coluna 'zeros_ou_false' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas),
perf AS (
  SELECT stack(41,
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'num_remessa', 'string', COUNT_IF(`num_remessa` IS NULL), COUNT_IF(`num_remessa` IS NOT NULL AND lower(trim(`num_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_remessa`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'num_item_remessa', 'string', COUNT_IF(`num_item_remessa` IS NULL), COUNT_IF(`num_item_remessa` IS NOT NULL AND lower(trim(`num_item_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_remessa`) RLIKE '^0+([.,]0+)?$'),
    'nm_material_cliente', 'string', COUNT_IF(`nm_material_cliente` IS NULL), COUNT_IF(`nm_material_cliente` IS NOT NULL AND lower(trim(`nm_material_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_material_cliente`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'date', COUNT_IF(`dt_criacao` IS NULL), 0L, 0L,
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'nm_fornecedor', 'string', COUNT_IF(`nm_fornecedor` IS NULL), COUNT_IF(`nm_fornecedor` IS NOT NULL AND lower(trim(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'sg_unidade_medida_basica', 'string', COUNT_IF(`sg_unidade_medida_basica` IS NULL), COUNT_IF(`sg_unidade_medida_basica` IS NOT NULL AND lower(trim(`sg_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'qt_fornecida_fato_umv', 'decimal(18,3)', COUNT_IF(`qt_fornecida_fato_umv` IS NULL), 0L, COUNT_IF(`qt_fornecida_fato_umv` = 0),
    'num_pedido_compra', 'string', COUNT_IF(`num_pedido_compra` IS NULL), COUNT_IF(`num_pedido_compra` IS NOT NULL AND lower(trim(`num_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'nm_inverted', 'string', COUNT_IF(`nm_inverted` IS NULL), COUNT_IF(`nm_inverted` IS NOT NULL AND lower(trim(`nm_inverted`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_inverted`) RLIKE '^0+([.,]0+)?$'),
    'qt_total_volumes_remessa', 'int', COUNT_IF(`qt_total_volumes_remessa` IS NULL), 0L, COUNT_IF(`qt_total_volumes_remessa` = 0),
    'qt_itens_relevantes', 'int', COUNT_IF(`qt_itens_relevantes` IS NULL), 0L, COUNT_IF(`qt_itens_relevantes` = 0),
    'cod_local_expedicao_ponto_recebimento', 'string', COUNT_IF(`cod_local_expedicao_ponto_recebimento` IS NULL), COUNT_IF(`cod_local_expedicao_ponto_recebimento` IS NOT NULL AND lower(trim(`cod_local_expedicao_ponto_recebimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_local_expedicao_ponto_recebimento`) RLIKE '^0+([.,]0+)?$'),
    'num_conhecimento_transporte', 'string', COUNT_IF(`num_conhecimento_transporte` IS NULL), COUNT_IF(`num_conhecimento_transporte` IS NOT NULL AND lower(trim(`num_conhecimento_transporte`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_conhecimento_transporte`) RLIKE '^0+([.,]0+)?$'),
    'qt_peso_bruto_total', 'decimal(18,3)', COUNT_IF(`qt_peso_bruto_total` IS NULL), 0L, COUNT_IF(`qt_peso_bruto_total` = 0),
    'st_verificacao_credito_global', 'string', COUNT_IF(`st_verificacao_credito_global` IS NULL), COUNT_IF(`st_verificacao_credito_global` IS NOT NULL AND lower(trim(`st_verificacao_credito_global`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_verificacao_credito_global`) RLIKE '^0+([.,]0+)?$'),
    'ind_correcao_re_avancada', 'boolean', COUNT_IF(`ind_correcao_re_avancada` IS NULL), 0L, COUNT_IF(`ind_correcao_re_avancada` = false),
    'qt_contador', 'int', COUNT_IF(`qt_contador` IS NULL), 0L, COUNT_IF(`qt_contador` = 0),
    'ind_item_devolucao', 'string', COUNT_IF(`ind_item_devolucao` IS NULL), COUNT_IF(`ind_item_devolucao` IS NOT NULL AND lower(trim(`ind_item_devolucao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_devolucao`) RLIKE '^0+([.,]0+)?$'),
    'tp_remessa', 'string', COUNT_IF(`tp_remessa` IS NULL), COUNT_IF(`tp_remessa` IS NOT NULL AND lower(trim(`tp_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_remessa`) RLIKE '^0+([.,]0+)?$'),
    'tp_ctg_item_remessa', 'string', COUNT_IF(`tp_ctg_item_remessa` IS NULL), COUNT_IF(`tp_ctg_item_remessa` IS NOT NULL AND lower(trim(`tp_ctg_item_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_ctg_item_remessa`) RLIKE '^0+([.,]0+)?$'),
    'tp_movimento_administracao_estoques', 'string', COUNT_IF(`tp_movimento_administracao_estoques` IS NULL), COUNT_IF(`tp_movimento_administracao_estoques` IS NOT NULL AND lower(trim(`tp_movimento_administracao_estoques`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_movimento_administracao_estoques`) RLIKE '^0+([.,]0+)?$'),
    'st_movimento_mercadorias_item', 'string', COUNT_IF(`st_movimento_mercadorias_item` IS NULL), COUNT_IF(`st_movimento_mercadorias_item` IS NOT NULL AND lower(trim(`st_movimento_mercadorias_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_movimento_mercadorias_item`) RLIKE '^0+([.,]0+)?$'),
    'dt_remessa', 'date', COUNT_IF(`dt_remessa` IS NULL), 0L, 0L,
    'dt_movimentacao_mercadoria_real', 'date', COUNT_IF(`dt_movimentacao_mercadoria_real` IS NULL), 0L, 0L,
    'dt_planejada_movimento_mercadoria', 'date', COUNT_IF(`dt_planejada_movimento_mercadoria` IS NULL), 0L, 0L,
    'dt_disponibilizacao_material', 'date', COUNT_IF(`dt_disponibilizacao_material` IS NULL), 0L, 0L,
    'num_item_pedido_compra', 'string', COUNT_IF(`num_item_pedido_compra` IS NULL), COUNT_IF(`num_item_pedido_compra` IS NOT NULL AND lower(trim(`num_item_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_referencia', 'string', COUNT_IF(`num_referencia` IS NULL), COUNT_IF(`num_referencia` IS NOT NULL AND lower(trim(`num_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_referencia`) RLIKE '^0+([.,]0+)?$'),
    'st_movimento_mercadorias_todos_itens', 'string', COUNT_IF(`st_movimento_mercadorias_todos_itens` IS NULL), COUNT_IF(`st_movimento_mercadorias_todos_itens` IS NOT NULL AND lower(trim(`st_movimento_mercadorias_todos_itens`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_movimento_mercadorias_todos_itens`) RLIKE '^0+([.,]0+)?$'),
    'cod_zonas_disponibilizacao_sistema_deposito', 'string', COUNT_IF(`cod_zonas_disponibilizacao_sistema_deposito` IS NULL), COUNT_IF(`cod_zonas_disponibilizacao_sistema_deposito` IS NOT NULL AND lower(trim(`cod_zonas_disponibilizacao_sistema_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_zonas_disponibilizacao_sistema_deposito`) RLIKE '^0+([.,]0+)?$'),
    'cod_organizacao_vendas', 'string', COUNT_IF(`cod_organizacao_vendas` IS NULL), COUNT_IF(`cod_organizacao_vendas` IS NOT NULL AND lower(trim(`cod_organizacao_vendas`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_organizacao_vendas`) RLIKE '^0+([.,]0+)?$'),
    'cod_canal_distribuicao', 'string', COUNT_IF(`cod_canal_distribuicao` IS NULL), COUNT_IF(`cod_canal_distribuicao` IS NOT NULL AND lower(trim(`cod_canal_distribuicao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_canal_distribuicao`) RLIKE '^0+([.,]0+)?$'),
    'cod_setor_atividade_organizacao', 'string', COUNT_IF(`cod_setor_atividade_organizacao` IS NULL), COUNT_IF(`cod_setor_atividade_organizacao` IS NOT NULL AND lower(trim(`cod_setor_atividade_organizacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_setor_atividade_organizacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_equipe_vendas', 'string', COUNT_IF(`tp_equipe_vendas` IS NULL), COUNT_IF(`tp_equipe_vendas` IS NOT NULL AND lower(trim(`tp_equipe_vendas`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_equipe_vendas`) RLIKE '^0+([.,]0+)?$'),
    'cod_emissor_ordem', 'string', COUNT_IF(`cod_emissor_ordem` IS NULL), COUNT_IF(`cod_emissor_ordem` IS NOT NULL AND lower(trim(`cod_emissor_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_emissor_ordem`) RLIKE '^0+([.,]0+)?$'),
    'dh_atualizacao', 'timestamp', COUNT_IF(`dh_atualizacao` IS NULL), 0L, 0L,
    'dh_carga', 'timestamp', COUNT_IF(`dh_carga` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas),
card AS (
  SELECT stack(41,
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'num_remessa', 'string', approx_count_distinct(`num_remessa`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'num_item_remessa', 'string', approx_count_distinct(`num_item_remessa`),
    'nm_material_cliente', 'string', approx_count_distinct(`nm_material_cliente`),
    'dt_criacao', 'date', approx_count_distinct(`dt_criacao`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'nm_fornecedor', 'string', approx_count_distinct(`nm_fornecedor`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'sg_unidade_medida_basica', 'string', approx_count_distinct(`sg_unidade_medida_basica`),
    'qt_fornecida_fato_umv', 'decimal(18,3)', approx_count_distinct(`qt_fornecida_fato_umv`),
    'num_pedido_compra', 'string', approx_count_distinct(`num_pedido_compra`),
    'nm_inverted', 'string', approx_count_distinct(`nm_inverted`),
    'qt_total_volumes_remessa', 'int', approx_count_distinct(`qt_total_volumes_remessa`),
    'qt_itens_relevantes', 'int', approx_count_distinct(`qt_itens_relevantes`),
    'cod_local_expedicao_ponto_recebimento', 'string', approx_count_distinct(`cod_local_expedicao_ponto_recebimento`),
    'num_conhecimento_transporte', 'string', approx_count_distinct(`num_conhecimento_transporte`),
    'qt_peso_bruto_total', 'decimal(18,3)', approx_count_distinct(`qt_peso_bruto_total`),
    'st_verificacao_credito_global', 'string', approx_count_distinct(`st_verificacao_credito_global`),
    'ind_correcao_re_avancada', 'boolean', approx_count_distinct(`ind_correcao_re_avancada`),
    'qt_contador', 'int', approx_count_distinct(`qt_contador`),
    'ind_item_devolucao', 'string', approx_count_distinct(`ind_item_devolucao`),
    'tp_remessa', 'string', approx_count_distinct(`tp_remessa`),
    'tp_ctg_item_remessa', 'string', approx_count_distinct(`tp_ctg_item_remessa`),
    'tp_movimento_administracao_estoques', 'string', approx_count_distinct(`tp_movimento_administracao_estoques`),
    'st_movimento_mercadorias_item', 'string', approx_count_distinct(`st_movimento_mercadorias_item`),
    'dt_remessa', 'date', approx_count_distinct(`dt_remessa`),
    'dt_movimentacao_mercadoria_real', 'date', approx_count_distinct(`dt_movimentacao_mercadoria_real`),
    'dt_planejada_movimento_mercadoria', 'date', approx_count_distinct(`dt_planejada_movimento_mercadoria`),
    'dt_disponibilizacao_material', 'date', approx_count_distinct(`dt_disponibilizacao_material`),
    'num_item_pedido_compra', 'string', approx_count_distinct(`num_item_pedido_compra`),
    'num_referencia', 'string', approx_count_distinct(`num_referencia`),
    'st_movimento_mercadorias_todos_itens', 'string', approx_count_distinct(`st_movimento_mercadorias_todos_itens`),
    'cod_zonas_disponibilizacao_sistema_deposito', 'string', approx_count_distinct(`cod_zonas_disponibilizacao_sistema_deposito`),
    'cod_organizacao_vendas', 'string', approx_count_distinct(`cod_organizacao_vendas`),
    'cod_canal_distribuicao', 'string', approx_count_distinct(`cod_canal_distribuicao`),
    'cod_setor_atividade_organizacao', 'string', approx_count_distinct(`cod_setor_atividade_organizacao`),
    'tp_equipe_vendas', 'string', approx_count_distinct(`tp_equipe_vendas`),
    'cod_emissor_ordem', 'string', approx_count_distinct(`cod_emissor_ordem`),
    'dh_atualizacao', 'timestamp', approx_count_distinct(`dh_atualizacao`),
    'dh_carga', 'timestamp', approx_count_distinct(`dh_carga`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS
(SELECT 'tp_remessa' AS coluna, CAST(`tp_remessa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `tp_remessa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_ctg_item_remessa' AS coluna, CAST(`tp_ctg_item_remessa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `tp_ctg_item_remessa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_movimento_administracao_estoques' AS coluna, CAST(`tp_movimento_administracao_estoques` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `tp_movimento_administracao_estoques` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_movimento_mercadorias_item' AS coluna, CAST(`st_movimento_mercadorias_item` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `st_movimento_mercadorias_item` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_movimento_mercadorias_todos_itens' AS coluna, CAST(`st_movimento_mercadorias_todos_itens` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `st_movimento_mercadorias_todos_itens` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_verificacao_credito_global' AS coluna, CAST(`st_verificacao_credito_global` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `st_verificacao_credito_global` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_devolucao' AS coluna, CAST(`ind_item_devolucao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `ind_item_devolucao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_medida_basica' AS coluna, CAST(`sg_unidade_medida_basica` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `sg_unidade_medida_basica` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_organizacao_vendas' AS coluna, CAST(`cod_organizacao_vendas` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `cod_organizacao_vendas` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_canal_distribuicao' AS coluna, CAST(`cod_canal_distribuicao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `cod_canal_distribuicao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_local_expedicao_ponto_recebimento' AS coluna, CAST(`cod_local_expedicao_ponto_recebimento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas GROUP BY `cod_local_expedicao_ponto_recebimento` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Flags booleanas

A tabela tem 1 colunas `boolean`. Flag com valor único é candidata a
nunca ter sido carregada.

> **Atenção na comparação:** o SAP exporta indicadores como `X` / vazio;
> o Datalake grava `true` / `false`. É conversão de tipo, não divergência de valor.

In [0]:
-- FLAGS BOOLEANAS
SELECT 'ind_correcao_re_avancada' AS flag,
       COUNT_IF(`ind_correcao_re_avancada` = true)  AS verdadeiro,
       COUNT_IF(`ind_correcao_re_avancada` = false) AS falso,
       COUNT_IF(`ind_correcao_re_avancada` IS NULL) AS nulo,
       ROUND(100.0 * COUNT_IF(`ind_correcao_re_avancada` = true) / COUNT(*), 2) AS pct_verdadeiro
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
ORDER BY pct_verdadeiro DESC;

## 10. Perfil dos campos numéricos

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(5,
    'qt_fornecida_fato_umv', 'decimal(18,3)', COUNT(`qt_fornecida_fato_umv`), CAST(MIN(`qt_fornecida_fato_umv`) AS DOUBLE), CAST(MAX(`qt_fornecida_fato_umv`) AS DOUBLE), CAST(AVG(`qt_fornecida_fato_umv`) AS DOUBLE), CAST(percentile_approx(`qt_fornecida_fato_umv`, 0.5) AS DOUBLE), COUNT_IF(`qt_fornecida_fato_umv` < 0), COUNT_IF(`qt_fornecida_fato_umv` = 0),
    'qt_total_volumes_remessa', 'int', COUNT(`qt_total_volumes_remessa`), CAST(MIN(`qt_total_volumes_remessa`) AS DOUBLE), CAST(MAX(`qt_total_volumes_remessa`) AS DOUBLE), CAST(AVG(`qt_total_volumes_remessa`) AS DOUBLE), CAST(percentile_approx(`qt_total_volumes_remessa`, 0.5) AS DOUBLE), COUNT_IF(`qt_total_volumes_remessa` < 0), COUNT_IF(`qt_total_volumes_remessa` = 0),
    'qt_itens_relevantes', 'int', COUNT(`qt_itens_relevantes`), CAST(MIN(`qt_itens_relevantes`) AS DOUBLE), CAST(MAX(`qt_itens_relevantes`) AS DOUBLE), CAST(AVG(`qt_itens_relevantes`) AS DOUBLE), CAST(percentile_approx(`qt_itens_relevantes`, 0.5) AS DOUBLE), COUNT_IF(`qt_itens_relevantes` < 0), COUNT_IF(`qt_itens_relevantes` = 0),
    'qt_peso_bruto_total', 'decimal(18,3)', COUNT(`qt_peso_bruto_total`), CAST(MIN(`qt_peso_bruto_total`) AS DOUBLE), CAST(MAX(`qt_peso_bruto_total`) AS DOUBLE), CAST(AVG(`qt_peso_bruto_total`) AS DOUBLE), CAST(percentile_approx(`qt_peso_bruto_total`, 0.5) AS DOUBLE), COUNT_IF(`qt_peso_bruto_total` < 0), COUNT_IF(`qt_peso_bruto_total` = 0),
    'qt_contador', 'int', COUNT(`qt_contador`), CAST(MIN(`qt_contador`) AS DOUBLE), CAST(MAX(`qt_contador`) AS DOUBLE), CAST(AVG(`qt_contador`) AS DOUBLE), CAST(percentile_approx(`qt_contador`, 0.5) AS DOUBLE), COUNT_IF(`qt_contador` < 0), COUNT_IF(`qt_contador` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(5,
    'qt_fornecida_fato_umv', CAST(SUM(`qt_fornecida_fato_umv`) AS DOUBLE), CAST(ROUND(SUM(`qt_fornecida_fato_umv`), 2) AS STRING), COUNT(`qt_fornecida_fato_umv`),
    'qt_total_volumes_remessa', CAST(SUM(`qt_total_volumes_remessa`) AS DOUBLE), CAST(ROUND(SUM(`qt_total_volumes_remessa`), 2) AS STRING), COUNT(`qt_total_volumes_remessa`),
    'qt_itens_relevantes', CAST(SUM(`qt_itens_relevantes`) AS DOUBLE), CAST(ROUND(SUM(`qt_itens_relevantes`), 2) AS STRING), COUNT(`qt_itens_relevantes`),
    'qt_peso_bruto_total', CAST(SUM(`qt_peso_bruto_total`) AS DOUBLE), CAST(ROUND(SUM(`qt_peso_bruto_total`), 2) AS STRING), COUNT(`qt_peso_bruto_total`),
    'qt_contador', CAST(SUM(`qt_contador`) AS DOUBLE), CAST(ROUND(SUM(`qt_contador`), 2) AS STRING), COUNT(`qt_contador`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
)
ORDER BY coluna;

## 12. Datas em tipo nativo

In [0]:
-- DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(5,
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL), CAST(MIN(`dt_criacao`) AS STRING), CAST(MAX(`dt_criacao`) AS STRING), COUNT(DISTINCT `dt_criacao`), COUNT_IF(`dt_criacao` > current_date()),
    'dt_remessa', COUNT_IF(`dt_remessa` IS NULL), CAST(MIN(`dt_remessa`) AS STRING), CAST(MAX(`dt_remessa`) AS STRING), COUNT(DISTINCT `dt_remessa`), COUNT_IF(`dt_remessa` > current_date()),
    'dt_movimentacao_mercadoria_real', COUNT_IF(`dt_movimentacao_mercadoria_real` IS NULL), CAST(MIN(`dt_movimentacao_mercadoria_real`) AS STRING), CAST(MAX(`dt_movimentacao_mercadoria_real`) AS STRING), COUNT(DISTINCT `dt_movimentacao_mercadoria_real`), COUNT_IF(`dt_movimentacao_mercadoria_real` > current_date()),
    'dt_planejada_movimento_mercadoria', COUNT_IF(`dt_planejada_movimento_mercadoria` IS NULL), CAST(MIN(`dt_planejada_movimento_mercadoria`) AS STRING), CAST(MAX(`dt_planejada_movimento_mercadoria`) AS STRING), COUNT(DISTINCT `dt_planejada_movimento_mercadoria`), COUNT_IF(`dt_planejada_movimento_mercadoria` > current_date()),
    'dt_disponibilizacao_material', COUNT_IF(`dt_disponibilizacao_material` IS NULL), CAST(MIN(`dt_disponibilizacao_material`) AS STRING), CAST(MAX(`dt_disponibilizacao_material`) AS STRING), COUNT(DISTINCT `dt_disponibilizacao_material`), COUNT_IF(`dt_disponibilizacao_material` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
)
ORDER BY coluna;

## 12.2 Timestamps

In [0]:
-- TIMESTAMPS
SELECT * FROM (
  SELECT stack(2,
    'dh_atualizacao', COUNT_IF(`dh_atualizacao` IS NULL), CAST(MIN(`dh_atualizacao`) AS STRING), CAST(MAX(`dh_atualizacao`) AS STRING), COUNT(DISTINCT `dh_atualizacao`),
    'dh_carga', COUNT_IF(`dh_carga` IS NULL), CAST(MIN(`dh_carga`) AS STRING), CAST(MAX(`dh_carga`) AS STRING), COUNT(DISTINCT `dh_carga`)
  ) AS (coluna, nulos, minimo, maximo, distintos)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(9,
    'num_remessa', 'string', COUNT_IF(CAST(`num_remessa` AS STRING) IS NULL OR trim(CAST(`num_remessa` AS STRING)) = ''), MIN(length(trim(CAST(`num_remessa` AS STRING)))), MAX(length(trim(CAST(`num_remessa` AS STRING)))), COUNT_IF(trim(CAST(`num_remessa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_remessa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_remessa` AS STRING)), '^0+', '')),
    'num_item_remessa', 'string', COUNT_IF(CAST(`num_item_remessa` AS STRING) IS NULL OR trim(CAST(`num_item_remessa` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_remessa` AS STRING)))), MAX(length(trim(CAST(`num_item_remessa` AS STRING)))), COUNT_IF(trim(CAST(`num_item_remessa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_remessa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_remessa` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'num_pedido_compra', 'string', COUNT_IF(CAST(`num_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '')),
    'num_item_pedido_compra', 'string', COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_item_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '')),
    'cod_deposito', 'string', COUNT_IF(CAST(`cod_deposito` AS STRING) IS NULL OR trim(CAST(`cod_deposito` AS STRING)) = ''), MIN(length(trim(CAST(`cod_deposito` AS STRING)))), MAX(length(trim(CAST(`cod_deposito` AS STRING)))), COUNT_IF(trim(CAST(`cod_deposito` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_deposito` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_deposito` AS STRING)), '^0+', '')),
    'num_referencia', 'string', COUNT_IF(CAST(`num_referencia` AS STRING) IS NULL OR trim(CAST(`num_referencia` AS STRING)) = ''), MIN(length(trim(CAST(`num_referencia` AS STRING)))), MAX(length(trim(CAST(`num_referencia` AS STRING)))), COUNT_IF(trim(CAST(`num_referencia` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_referencia` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_referencia` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real,
não granularidade adicional.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`num_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`num_item_remessa` AS STRING), ''), COALESCE(CAST(`nm_material_cliente` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_fornecida_fato_umv` AS STRING), ''), COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`nm_inverted` AS STRING), ''), COALESCE(CAST(`qt_total_volumes_remessa` AS STRING), ''), COALESCE(CAST(`qt_itens_relevantes` AS STRING), ''), COALESCE(CAST(`cod_local_expedicao_ponto_recebimento` AS STRING), ''), COALESCE(CAST(`num_conhecimento_transporte` AS STRING), ''), COALESCE(CAST(`qt_peso_bruto_total` AS STRING), ''), COALESCE(CAST(`st_verificacao_credito_global` AS STRING), ''), COALESCE(CAST(`ind_correcao_re_avancada` AS STRING), ''), COALESCE(CAST(`qt_contador` AS STRING), ''), COALESCE(CAST(`ind_item_devolucao` AS STRING), ''), COALESCE(CAST(`tp_remessa` AS STRING), ''), COALESCE(CAST(`tp_ctg_item_remessa` AS STRING), ''), COALESCE(CAST(`tp_movimento_administracao_estoques` AS STRING), ''), COALESCE(CAST(`st_movimento_mercadorias_item` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_movimentacao_mercadoria_real` AS STRING), ''), COALESCE(CAST(`dt_planejada_movimento_mercadoria` AS STRING), ''), COALESCE(CAST(`dt_disponibilizacao_material` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`st_movimento_mercadorias_todos_itens` AS STRING), ''), COALESCE(CAST(`cod_zonas_disponibilizacao_sistema_deposito` AS STRING), ''), COALESCE(CAST(`cod_organizacao_vendas` AS STRING), ''), COALESCE(CAST(`cod_canal_distribuicao` AS STRING), ''), COALESCE(CAST(`cod_setor_atividade_organizacao` AS STRING), ''), COALESCE(CAST(`tp_equipe_vendas` AS STRING), ''), COALESCE(CAST(`cod_emissor_ordem` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`num_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`num_item_remessa` AS STRING), ''), COALESCE(CAST(`nm_material_cliente` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_fornecida_fato_umv` AS STRING), ''), COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`nm_inverted` AS STRING), ''), COALESCE(CAST(`qt_total_volumes_remessa` AS STRING), ''), COALESCE(CAST(`qt_itens_relevantes` AS STRING), ''), COALESCE(CAST(`cod_local_expedicao_ponto_recebimento` AS STRING), ''), COALESCE(CAST(`num_conhecimento_transporte` AS STRING), ''), COALESCE(CAST(`qt_peso_bruto_total` AS STRING), ''), COALESCE(CAST(`st_verificacao_credito_global` AS STRING), ''), COALESCE(CAST(`ind_correcao_re_avancada` AS STRING), ''), COALESCE(CAST(`qt_contador` AS STRING), ''), COALESCE(CAST(`ind_item_devolucao` AS STRING), ''), COALESCE(CAST(`tp_remessa` AS STRING), ''), COALESCE(CAST(`tp_ctg_item_remessa` AS STRING), ''), COALESCE(CAST(`tp_movimento_administracao_estoques` AS STRING), ''), COALESCE(CAST(`st_movimento_mercadorias_item` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_movimentacao_mercadoria_real` AS STRING), ''), COALESCE(CAST(`dt_planejada_movimento_mercadoria` AS STRING), ''), COALESCE(CAST(`dt_disponibilizacao_material` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`st_movimento_mercadorias_todos_itens` AS STRING), ''), COALESCE(CAST(`cod_zonas_disponibilizacao_sistema_deposito` AS STRING), ''), COALESCE(CAST(`cod_organizacao_vendas` AS STRING), ''), COALESCE(CAST(`cod_canal_distribuicao` AS STRING), ''), COALESCE(CAST(`cod_setor_atividade_organizacao` AS STRING), ''), COALESCE(CAST(`tp_equipe_vendas` AS STRING), ''), COALESCE(CAST(`cod_emissor_ordem` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`num_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`num_item_remessa` AS STRING), ''), COALESCE(CAST(`nm_material_cliente` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`sg_unidade_medida_basica` AS STRING), ''), COALESCE(CAST(`qt_fornecida_fato_umv` AS STRING), ''), COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`nm_inverted` AS STRING), ''), COALESCE(CAST(`qt_total_volumes_remessa` AS STRING), ''), COALESCE(CAST(`qt_itens_relevantes` AS STRING), ''), COALESCE(CAST(`cod_local_expedicao_ponto_recebimento` AS STRING), ''), COALESCE(CAST(`num_conhecimento_transporte` AS STRING), ''), COALESCE(CAST(`qt_peso_bruto_total` AS STRING), ''), COALESCE(CAST(`st_verificacao_credito_global` AS STRING), ''), COALESCE(CAST(`ind_correcao_re_avancada` AS STRING), ''), COALESCE(CAST(`qt_contador` AS STRING), ''), COALESCE(CAST(`ind_item_devolucao` AS STRING), ''), COALESCE(CAST(`tp_remessa` AS STRING), ''), COALESCE(CAST(`tp_ctg_item_remessa` AS STRING), ''), COALESCE(CAST(`tp_movimento_administracao_estoques` AS STRING), ''), COALESCE(CAST(`st_movimento_mercadorias_item` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_movimentacao_mercadoria_real` AS STRING), ''), COALESCE(CAST(`dt_planejada_movimento_mercadoria` AS STRING), ''), COALESCE(CAST(`dt_disponibilizacao_material` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_referencia` AS STRING), ''), COALESCE(CAST(`st_movimento_mercadorias_todos_itens` AS STRING), ''), COALESCE(CAST(`cod_zonas_disponibilizacao_sistema_deposito` AS STRING), ''), COALESCE(CAST(`cod_organizacao_vendas` AS STRING), ''), COALESCE(CAST(`cod_canal_distribuicao` AS STRING), ''), COALESCE(CAST(`cod_setor_atividade_organizacao` AS STRING), ''), COALESCE(CAST(`tp_equipe_vendas` AS STRING), ''), COALESCE(CAST(`cod_emissor_ordem` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas;

## 15. Amostra de linhas completas

In [0]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
ORDER BY `num_remessa`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_remessa
SELECT COALESCE(NULLIF(trim(CAST(`tp_remessa` AS STRING)), ''), '(vazio)') AS `tp_remessa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_remessa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_movimento_administracao_estoques
SELECT COALESCE(NULLIF(trim(CAST(`tp_movimento_administracao_estoques` AS STRING)), ''), '(vazio)') AS `tp_movimento_administracao_estoques`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_movimento_administracao_estoques` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR st_movimento_mercadorias_todos_itens
SELECT COALESCE(NULLIF(trim(CAST(`st_movimento_mercadorias_todos_itens` AS STRING)), ''), '(vazio)') AS `st_movimento_mercadorias_todos_itens`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY COALESCE(NULLIF(trim(CAST(`st_movimento_mercadorias_todos_itens` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_deposito
SELECT COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)') AS `cod_deposito`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas), 2) AS pct
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [0]:
-- 17. FRESHNESS
-- Esta tabela nao possui coluna de data de ingestao. Use os timestamps de carga.
SELECT MIN(dh_carga)              AS carga_min,
       MAX(dh_carga)              AS carga_max,
       COUNT(DISTINCT to_date(dh_carga)) AS dias_de_carga,
       MIN(dh_atualizacao)        AS atualizacao_min,
       MAX(dh_atualizacao)        AS atualizacao_max
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas;

## 18. Análises específicas — VL06IF

### 18.1 Achatamento no primeiro item — MEDIÇÃO DA PERDA

**O achado mais importante desta view.**

Cerca de 12 colunas trazem o comentário *"Primeiro item"*: a view mantém **uma linha por remessa**
e descarta os demais itens. Mas `qt_itens_relevantes` guarda o `COUNT` de itens da remessa original.

Isso permite **medir exatamente** quantos itens foram descartados — algo que normalmente
só se descobre comparando com o SAP.

In [0]:
-- 18.1 QUANTOS ITENS FORAM DESCARTADOS
SELECT COUNT(*)                                             AS linhas_na_view,
       SUM(COALESCE(qt_itens_relevantes, 0))                AS itens_no_sap_original,
       SUM(COALESCE(qt_itens_relevantes, 0)) - COUNT(*)     AS itens_DESCARTADOS,
       ROUND(100.0 * (SUM(COALESCE(qt_itens_relevantes, 0)) - COUNT(*))
                   / NULLIF(SUM(COALESCE(qt_itens_relevantes, 0)), 0), 2) AS pct_descartado,
       COUNT_IF(qt_itens_relevantes > 1)                    AS remessas_com_varios_itens,
       COUNT_IF(qt_itens_relevantes = 1)                    AS remessas_com_item_unico,
       COUNT_IF(qt_itens_relevantes IS NULL OR qt_itens_relevantes = 0) AS sem_contagem,
       MAX(qt_itens_relevantes)                             AS maior_qtd_itens
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas;

In [0]:
-- 18.1b DISTRIBUICAO DE ITENS POR REMESSA
SELECT qt_itens_relevantes AS itens_na_remessa,
       COUNT(*)            AS remessas,
       SUM(qt_itens_relevantes) - COUNT(*) AS itens_perdidos_neste_grupo
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY qt_itens_relevantes
ORDER BY qt_itens_relevantes
LIMIT 40;

### 18.2 Inbound vs outbound

O comentário declara que várias datas são nulas em ~95% dos casos inbound.
Esta célula separa os dois fluxos pelo tipo de movimento (101=GR inbound, 601=GI outbound).

In [0]:
-- 18.2 INBOUND x OUTBOUND
SELECT tp_movimento_administracao_estoques AS tipo_movimento,
       COUNT(*)                                              AS remessas,
       COUNT_IF(dt_remessa IS NULL)                          AS sem_dt_remessa,
       COUNT_IF(dt_movimentacao_mercadoria_real IS NULL)     AS sem_dt_movimentacao_real,
       COUNT_IF(dt_planejada_movimento_mercadoria IS NULL)   AS sem_dt_planejada,
       COUNT_IF(dt_disponibilizacao_material IS NULL)        AS sem_dt_disponibilizacao,
       ROUND(100.0 * COUNT_IF(dt_remessa IS NULL) / COUNT(*), 1) AS pct_sem_dt_remessa
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY tp_movimento_administracao_estoques
ORDER BY remessas DESC;

### 18.3 Colunas sem comentário no schema

`nm_inverted` e `qt_contador` são as únicas colunas **sem descrição** no `DESCRIBE`.
Essa é a assinatura típica de coluna nunca carregada.

In [0]:
-- 18.3 COLUNAS SEM COMENTARIO
SELECT 'nm_inverted' AS coluna,
       COUNT_IF(nm_inverted IS NOT NULL AND trim(nm_inverted) <> '') AS uteis,
       COUNT(DISTINCT nm_inverted)                                    AS distintos,
       COUNT(*)                                                       AS total,
       CASE WHEN COUNT_IF(nm_inverted IS NOT NULL AND trim(nm_inverted) <> '') = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END             AS veredito
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
UNION ALL
SELECT 'qt_contador',
       COUNT_IF(qt_contador IS NOT NULL),
       COUNT(DISTINCT qt_contador),
       COUNT(*),
       CASE WHEN COUNT_IF(qt_contador IS NOT NULL) = 0
            THEN 'NUNCA CARREGADA' ELSE 'possui dado' END
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas;

### 18.4 Status de movimento de mercadorias

Compara o status do **primeiro item** (`WBSTA`) com o status **geral da remessa** (`WBSTK`).
Divergência é esperada quando a remessa tem vários itens em estágios diferentes —
e reforça o problema de achatamento.

In [0]:
-- 18.4 STATUS: PRIMEIRO ITEM x REMESSA TODA
SELECT COALESCE(NULLIF(trim(st_movimento_mercadorias_item), ''), '(vazio)')       AS status_primeiro_item,
       COALESCE(NULLIF(trim(st_movimento_mercadorias_todos_itens), ''), '(vazio)') AS status_remessa,
       COUNT(*)                          AS remessas,
       ROUND(AVG(qt_itens_relevantes), 2) AS media_itens
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
GROUP BY COALESCE(NULLIF(trim(st_movimento_mercadorias_item), ''), '(vazio)'),
         COALESCE(NULLIF(trim(st_movimento_mercadorias_todos_itens), ''), '(vazio)')
ORDER BY remessas DESC
LIMIT 30;

### 18.5 Coerência do fluxo de datas

Sequência esperada: criação → disponibilização → remessa planejada → movimentação real.

In [0]:
-- 18.5 COERENCIA DO FLUXO DE DATAS
SELECT 'criacao <= dt_remessa' AS regra,
       COUNT_IF(dt_criacao IS NOT NULL AND dt_remessa IS NOT NULL) AS avaliadas,
       COUNT_IF(dt_criacao > dt_remessa)                            AS violacoes
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
UNION ALL
SELECT 'planejada <= real',
       COUNT_IF(dt_planejada_movimento_mercadoria IS NOT NULL
            AND dt_movimentacao_mercadoria_real IS NOT NULL),
       COUNT_IF(dt_planejada_movimento_mercadoria > dt_movimentacao_mercadoria_real)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
UNION ALL
SELECT 'criacao <= disponibilizacao',
       COUNT_IF(dt_criacao IS NOT NULL AND dt_disponibilizacao_material IS NOT NULL),
       COUNT_IF(dt_criacao > dt_disponibilizacao_material)
  FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
ORDER BY violacoes DESC;

### 18.6 Vínculo com pedido de compra

In [0]:
-- 18.6 VINCULO COM PEDIDO DE COMPRA
SELECT COUNT(*)                                                    AS remessas,
       COUNT_IF(num_pedido_compra IS NOT NULL
            AND trim(num_pedido_compra) <> '')                     AS com_pedido,
       COUNT_IF(num_pedido_compra IS NULL OR trim(num_pedido_compra) = '') AS sem_pedido,
       COUNT_IF(cod_fornecedor IS NOT NULL AND trim(cod_fornecedor) <> '') AS com_fornecedor,
       COUNT(DISTINCT num_pedido_compra)                           AS pedidos_distintos,
       COUNT_IF(ind_item_devolucao = 'X')                          AS itens_devolucao
FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas
ORDER BY `num_remessa`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas),
g AS (
  SELECT 'num_remessa' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_remessa` FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas)
UNION ALL
  SELECT 'num_remessa + num_item_remessa' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_remessa`, `num_item_remessa` FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas)
UNION ALL
  SELECT 'num_remessa + cod_centro' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_remessa`, `cod_centro` FROM dev_procurement.corp_curated.vw_ds_log_monitor_remessas)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'VL06IF' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'vw_ds_log_monitor_remessas'
UNION ALL SELECT 'IDENTIFICACAO', 'tipo de objeto', 'VIEW'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '41'
UNION ALL SELECT 'IDENTIFICACAO', 'coluna de recorte', 'cod_centro'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Gerar o notebook de cenário com esse recorte.
3. Extrair a VL06IF no SAP com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] Booleanas: `true`/`false` no Datalake vs `X`/vazio no SAP (seção 9)
- [ ] Alertas de partida verificados (seção 18)
